# Local (pandas) baseline: 

This notebook is the single machine version of the analysis, meaning it uses Python with the pandas library on one computer, and it is the non Big Data approach that we compare against Spark. It reads the flight files for , cleans them, runs the five group by aggregations that answer our questions and records how long the run took and how much memory it used. At the end it writes a single HTML report, meaning one page that shows every table and chart, so that the results can be viewed without reading the code.

A flight is counted as delayed when its arrival delay is 15 minutes or more.

## 1. Setup and configuration
Here we import the libraries and set which monthly files this run will use.

In [1]:
import os, time, zipfile, tracemalloc
import pandas as pd

RAW_DIR = os.path.join("..", "Data", "Raw")
DURATION_LABEL = "1 month (Jan 2024)"
FILES = [
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip"
]
USECOLS = ["Year", "Month", "DayOfWeek", "Reporting_Airline", "Origin", "Dest", "CRSDepTime", "DepDelay", "ArrDelay", "ArrDelayMinutes", "Cancelled", "Distance", "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
print("Duration:", DURATION_LABEL)
print("Monthly files:", len(FILES))

Duration: 1 month (Jan 2024)
Monthly files: 1


## 2. Load the data
We read only the columns we need, meaning a small set rather than all 110, so that less memory is used. pandas keeps everything in memory at once, so the larger runs are expected to strain or run out of memory, and this is part of what we are trying to show.

In [2]:
def read_month(zip_path):
    """Read the single on-time CSV inside a BTS monthly zip."""
    with zipfile.ZipFile(zip_path) as z:
        csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(csv_name) as f:
            return pd.read_csv(f, usecols=USECOLS, low_memory=False)

tracemalloc.start()
t0 = time.perf_counter()
frames = []
for fn in FILES:
    path = os.path.join(RAW_DIR, fn)
    if not os.path.exists(path):
        print("MISSING:", fn, "-> download into Data/Raw"); continue
    frames.append(read_month(path))
df = pd.concat(frames, ignore_index=True)
load_secs = time.perf_counter() - t0
print(f"Loaded {len(df):,} rows in {load_secs:,.1f}s")

Loaded 547,271 rows in 2.4s


## 3. Clean the data
We remove cancelled flights, mark each flight as delayed or not (a delay of 15 minutes or more counts as delayed) and work out the scheduled departure hour and the route, meaning the origin joined to the destination.

In [3]:
df = df[df["Cancelled"] != 1].copy()
df["delayed"] = (df["ArrDelay"] >= 15).astype(int)
df["dep_hour"] = (df["CRSDepTime"] // 100).clip(0, 23)
df["route"] = df["Origin"] + "-" + df["Dest"]
print("Flights after cleaning:", f"{len(df):,}")

Flights after cleaning: 526,882


## 4. Analysis
We run five group by aggregations, meaning we sort the flights into groups and summarize each group, and each one answers one of our questions: the worst carriers, the worst airports, the worst routes, the delay by hour of day and the mix of delay causes by year.

In [4]:
# 4a. by carrier
by_carrier = (df.groupby("Reporting_Airline")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean"))
    .sort_values("pct_delayed", ascending=False))
by_carrier["pct_delayed"] *= 100
by_carrier.head(15)

,flights,avg_arr_delay,pct_delayed
Reporting_Airline,,,
AA,76098,19.174558,29.293805
B6,19246,13.522220,28.915099
MQ,19818,13.350355,27.616308
AS,14656,10.381649,27.258461
F9,14068,14.719331,27.111174
HA,6480,10.664710,26.820988
NK,20112,9.684378,26.223150
OH,15582,14.985842,25.182903
OO,54430,15.202837,24.034540


In [5]:
# 4b. worst airports (origin), min 1000 flights
by_airport = (df.groupby("Origin")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean")))
by_airport = by_airport[by_airport["flights"] >= 1000]
by_airport["pct_delayed"] *= 100
by_airport.sort_values("pct_delayed", ascending=False).head(15)

,flights,avg_arr_delay,pct_delayed
Origin,,,
GRR,1470,23.016360,33.197279
DSM,1151,29.726325,32.841008
ORD,18901,19.385470,32.151738
MIA,9956,16.614973,31.749699
DTW,9149,19.224195,30.768390
FLL,8195,13.506376,30.665040
HPN,1038,20.851064,30.539499
DFW,22916,17.168187,30.245243
BUF,1354,16.914941,29.837518


In [6]:
# 4c. worst routes, min 500 flights
by_route = (df.groupby("route")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean")))
by_route = by_route[by_route["flights"] >= 500]
by_route["pct_delayed"] *= 100
by_route.sort_values("pct_delayed", ascending=False).head(15)

,flights,avg_arr_delay,pct_delayed
route,,,
MIA-ATL,633,17.803487,33.333333
PHL-MCO,520,20.393064,33.076923
LAX-SFO,815,11.517791,33.006135
ATL-MIA,633,13.657188,32.227488
ORD-LGA,741,19.962162,31.578947
MCO-PHL,523,19.871893,30.783939
SJU-MCO,518,22.654440,30.694981
LGA-MIA,512,14.136986,30.664062
MIA-LGA,513,12.084479,29.629630


In [7]:
# 4d. by time of day (scheduled dep hour)
by_hour = (df.groupby("dep_hour")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean")))
by_hour["pct_delayed"] *= 100
by_hour

,flights,avg_arr_delay,pct_delayed
dep_hour,,,
0,738,6.639946,20.325203
1,333,16.084337,27.327327
2,124,3.548387,25.000000
3,125,3.960000,26.400000
4,58,1.172414,13.793103
5,13968,6.226598,15.392325
6,38361,3.142219,15.111702
7,35947,4.211118,17.219796
8,35428,4.888596,19.312408


In [8]:
# 4e. cause mix by year (share of delay minutes)
causes = ["CarrierDelay","WeatherDelay","NASDelay","SecurityDelay","LateAircraftDelay"]
cause_by_year = df.groupby("Year")[causes].sum()
cause_mix = cause_by_year.div(cause_by_year.sum(axis=1), axis=0) * 100
cause_mix

,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
Year,,,,,
2024,32.61895,10.267266,17.923831,0.237098,38.952855


## 5. Performance
Here we record the total runtime and the peak memory used, which are the numbers that we compare against Spark for the same data size.

In [9]:
total_secs = time.perf_counter() - t0
cur, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
print(f"Duration analysed : {DURATION_LABEL}")
print(f"Rows              : {len(df):,}")
print(f"Load time         : {load_secs:,.1f} s")
print(f"Total runtime     : {total_secs:,.1f} s")
print(f"Peak memory       : {peak/1e9:,.2f} GB")

Duration analysed : 1 month (Jan 2024)
Rows              : 526,882
Load time         : 2.4 s
Total runtime     : 3.2 s
Peak memory       : 0.22 GB


In [10]:
out = os.path.join("..", "Data", "Cleaned"); os.makedirs(out, exist_ok=True)
tag = DURATION_LABEL.split()[0]
by_carrier.to_csv(os.path.join(out, f"pandas_by_carrier_{tag}.csv"))
by_airport.sort_values("pct_delayed", ascending=False).to_csv(os.path.join(out, f"pandas_by_airport_{tag}.csv"))
by_route.sort_values("pct_delayed", ascending=False).to_csv(os.path.join(out, f"pandas_by_route_{tag}.csv"))
by_hour.to_csv(os.path.join(out, f"pandas_by_hour_{tag}.csv"))
cause_mix.to_csv(os.path.join(out, f"pandas_cause_mix_{tag}.csv"))
print("Saved result tables to Data/Cleaned")

Saved result tables to Data/Cleaned
